In [168]:
from pathlib import Path
import numpy as np
import pandas as pd
import shap
from tqdm import tqdm

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tabpfn import TabPFNClassifier
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import FunctionTransformer, StandardScaler, LabelEncoder, OneHotEncoder

from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score
import optuna
import shap

from sklearn.preprocessing import PolynomialFeatures
from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures

from meteocalc import dew_point

In [72]:
import torch
print(torch.cuda.is_available()) # Should return True

print(torch.__version__)

True
2.6.0+cu124


In [74]:
import warnings
warnings.filterwarnings("ignore")

In [75]:
DATA_DIR = Path("../../data/kaggle-irrigation")
SEED = 42

# Helper Funcs

In [171]:
def get_importances(model, X_test, y_test, model_name, has_builtin=True):
    """Compute permutation importance, plus gain-based importance if available."""
    # Permutation importance (works for any fitted model)
    perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
    df = pd.DataFrame({
        "feature": X_test.columns,
        f"{model_name}_perm": perm.importances_mean,
    })
    df[f"{model_name}_rank_perm"] = df[f"{model_name}_perm"].rank(ascending=False)

    # Gain-based importance (tree models only)
    if has_builtin:
        df[f"{model_name}_gain"] = model.feature_importances_
        df[f"{model_name}_rank_gain"] = df[f"{model_name}_gain"].rank(ascending=False)

    return df

#  Feature Engineering Pipeline

In [167]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import HDBSCAN
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

class HDBSCANFeature(BaseEstimator, TransformerMixin):
    def __init__(self, min_cluster_size=5, n_neighbors=5, append=True):
        self.min_cluster_size = min_cluster_size
        self.n_neighbors = n_neighbors
        self.append = append

    def fit(self, X, y=None):
        labels = HDBSCAN(min_cluster_size=self.min_cluster_size).fit_predict(X)
        # train a KNN to mimic the clustering on new data
        # (drop noise points from the KNN training set, or keep them as class -1)
        self.knn_ = KNeighborsClassifier(n_neighbors=self.n_neighbors).fit(X, labels)
        return self

    def transform(self, X):
        labels = self.knn_.predict(X).reshape(-1, 1).astype(float)
        return np.hstack([X, labels]) if self.append else labels

In [77]:
def add_features(d: pd.DataFrame):
    d = d.copy()

    d["heat_stress"]   = (d["Temperature_C"] * d["Sunlight_Hours"]).astype(int)
    d["drying_force"]  = (d["Wind_Speed_kmh"] * d["Temperature_C"] / (d["Humidity"] + 1)).astype(int)
    d["soil_quality"]  = (d["Organic_Carbon"] / (d["Electrical_Conductivity"] + 0.1)).astype(int)
    d["dew_point"] = d.apply(lambda row: dew_point(row["Temperature_C"], row["Humidity"]).c, axis=1)

    return d

In [78]:
feature_pipe = FunctionTransformer(add_features)

In [93]:
cat_selector = make_column_selector(dtype_include=["object", "category"])
num_selector = make_column_selector(dtype_include="number",)

cat_pipeline = Pipeline([
    ("rare_cat", RareLabelEncoder(tol=0.01)),
])

poly_ct = ColumnTransformer([
    ("poly", PolynomialFeatures(
        degree=3,
        interaction_only=True,
        include_bias=False),
        num_selector
     )
]).set_output(transform="pandas")

ct = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, cat_selector),
    ],
    remainder="passthrough",
).set_output(transform="pandas")

preprocessor_fe = Pipeline([
    ("add_features", feature_pipe),
    ("encode_and_discretize", ct),
    ("drop_consts", DropConstantFeatures()),
])

preprocessor_poly = Pipeline([
    ("add_features", feature_pipe),
    ("poly", poly_ct),
    ("encode_and_discretize", ct),
    ("drop_consts", DropConstantFeatures()),
])

# Get Training Data

In [94]:
df = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [95]:
X = df.drop(columns=["Irrigation_Need"])
y = df["Irrigation_Need"]

# convert dtypes for categorical handling
cat_cols = X.select_dtypes(include=["str", "object"]).columns
X[cat_cols] = X[cat_cols].astype("category")

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, shuffle=True, stratify=y, random_state=SEED)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# apply preprocessing
X_train_fe, X_test_fe = (preprocessor_fe.fit_transform(X_train), preprocessor_fe.fit_transform(X_test))
X_train_poly, X_test_poly = (preprocessor_poly.fit_transform(X_train), preprocessor_poly.fit_transform(X_test))

# num features

print(len(X_train.columns))
print(len(X_train_fe.columns))
print(len(X_train_poly.columns))

23
19


In [96]:
X_train.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
344420,Silt,5.66,51.38,0.46,1.46,33.21,66.34,775.35,6.23,17.09,Maize,Flowering,Zaid,Canal,Reservoir,13.63,Yes,66.62,Central
29017,Loamy,6.14,8.03,1.14,3.34,20.85,74.29,2374.11,7.52,1.94,Sugarcane,Vegetative,Rabi,Rainfed,Groundwater,5.56,Yes,92.24,West
570499,Sandy,5.77,59.04,1.08,0.55,14.44,37.99,2137.79,5.05,10.46,Rice,Sowing,Rabi,Drip,Reservoir,0.78,Yes,92.31,South
620696,Sandy,5.06,63.19,1.59,1.10,23.58,58.25,1226.78,4.54,1.15,Wheat,Harvest,Kharif,Canal,Reservoir,2.67,No,113.36,West
7445,Clay,6.28,35.94,1.11,1.57,34.30,71.04,2266.39,8.47,16.81,Maize,Vegetative,Kharif,Drip,Groundwater,6.74,Yes,96.90,North


In [83]:
X_train_fe.head()

,cat__Soil_Type,cat__Crop_Type,cat__Crop_Growth_Stage,cat__Season,cat__Irrigation_Type,cat__Water_Source,cat__Mulching_Used,cat__Region,num__Soil_pH,num__Soil_Moisture,...,num__Humidity,num__Rainfall_mm,num__Sunlight_Hours,num__Wind_Speed_kmh,num__Field_Area_hectare,num__Previous_Irrigation_mm,num__heat_stress,num__drying_force,num__soil_quality,num__dew_point
id,,,,,,,,,,,,,,,,,,,,,
344420,Silt,Maize,Flowering,Zaid,Canal,Reservoir,Yes,Central,1,3,...,2,0,1,4,4,2,2,4,0,3
29017,Loamy,Sugarcane,Vegetative,Rabi,Rainfed,Groundwater,Yes,West,1,0,...,3,4,2,0,1,3,1,0,0,2
570499,Sandy,Rice,Sowing,Rabi,Drip,Reservoir,Yes,South,1,4,...,0,4,0,2,0,3,0,1,0,0
620696,Sandy,Wheat,Harvest,Kharif,Canal,Reservoir,No,West,0,4,...,2,1,0,0,0,4,0,0,0,1
7445,Clay,Maize,Vegetative,Kharif,Drip,Groundwater,Yes,North,2,2,...,3,4,3,4,2,3,4,4,0,4


In [97]:
X_train_poly.head()

,num__poly__Soil_pH,num__poly__Soil_Moisture,num__poly__Organic_Carbon,num__poly__Electrical_Conductivity,num__poly__Temperature_C,num__poly__Humidity,num__poly__Rainfall_mm,num__poly__Sunlight_Hours,num__poly__Wind_Speed_kmh,num__poly__Field_Area_hectare,...,num__poly__Field_Area_hectare drying_force dew_point,num__poly__Field_Area_hectare soil_quality dew_point,num__poly__Previous_Irrigation_mm heat_stress drying_force,num__poly__Previous_Irrigation_mm heat_stress soil_quality,num__poly__Previous_Irrigation_mm heat_stress dew_point,num__poly__Previous_Irrigation_mm drying_force dew_point,num__poly__Previous_Irrigation_mm soil_quality dew_point,num__poly__heat_stress drying_force dew_point,num__poly__heat_stress soil_quality dew_point,num__poly__drying_force soil_quality dew_point
id,,,,,,,,,,,,,,,,,,,,,
344420,1,3,0,2,3,2,0,1,4,4,...,4,0,4,0,3,4,0,4,0,0
29017,1,0,3,4,1,3,4,2,0,1,...,0,0,0,0,3,0,0,0,0,0
570499,1,4,3,0,0,0,4,0,2,0,...,0,1,1,1,0,0,1,0,1,1
620696,0,4,4,1,1,2,1,0,0,0,...,0,2,0,1,2,0,2,0,2,0
7445,2,2,3,2,3,3,4,3,4,2,...,4,0,4,0,4,4,0,4,0,0


# Base Models

- uses all three feature sets for each model

## Light GBM

In [116]:
def train_lgbm(X_tr, X_te, y_tr, y_te, params = {}):
    lgbm = LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=0, **params )

    lgbm_cvs = cross_val_score(lgbm, X_tr, y_tr, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
    print(f"cv scores: {lgbm_cvs.mean():.4f} +/- {lgbm_cvs.std():.4f}")

    lgbm.fit(X_tr, y_tr)
    lgbm_preds = lgbm.predict(X_te)

    lgbm_metric_score = balanced_accuracy_score(y_te, lgbm_preds)
    print(f"Balanced Accuracy: {lgbm_metric_score:.4f}")

    return lgbm, lgbm_preds

### `X_train`

In [117]:
lgbm_base, lgbm_preds_base = train_lgbm(X_train, X_test, y_train, y_test)

cv scores: 0.9607 +/- 0.0013
Balanced Accuracy: 0.9645


### `X_train_fe`

In [118]:
lgbm_fe, lgbm_preds_fe = train_lgbm(X_train_fe, X_test_fe, y_train, y_test)

cv scores: 0.8431 +/- 0.0029
Balanced Accuracy: 0.8437


### `X_train_poly`

In [119]:
lgbm_poly, lgbm_preds_poly = train_lgbm(X_train_poly, X_test_poly, y_train, y_test)

cv scores: 0.6452 +/- 0.0025
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
Balanced Accuracy: 0.6515


## MLP

In [162]:
def train_mlp(X_tr, X_te, y_tr, y_te, params={}):
    # Early stopping defaults — override via params if needed
    es_defaults = {
        "early_stopping": True,
        "validation_fraction": 0.1,
        "n_iter_no_change": 10,
        "tol": 1e-4,
        "max_iter": 500,
    }

    es_defaults.update(params)

    mlp = MLPClassifier(
            random_state=SEED,
            hidden_layer_sizes=(32, 16),
            learning_rate="adaptive",
            solver="adam",
            **es_defaults
    )

    pipe = Pipeline([
        ("ct", ColumnTransformer(transformers=[
            ("ohe", OneHotEncoder(sparse_output=False), make_column_selector(dtype_include="category"))],
            remainder="passthrough").set_output(transform="pandas")),
        ("model", mlp)
    ])

    cvs = cross_val_score(
        pipe, X_tr, y_tr, cv=skf, n_jobs=-1, scoring="balanced_accuracy"
    )
    print(f"CV scores: {cvs.mean():.4f} +/- {cvs.std():.4f}")

    pipe.fit(X_tr, y_tr)
    preds = pipe.predict(X_te)

    score = balanced_accuracy_score(y_te, preds)
    print(f"Balanced Accuracy: {score:.4f}")

    return pipe, preds

### `X_train`

In [163]:
mlp_base, mlp_preds_base = train_mlp(X_train, X_test, y_train, y_test)

CV scores: 0.9382 +/- 0.0100
Balanced Accuracy: 0.9540


### `X_train_fe`

In [164]:
mlp_fe, mlp_preds_fe = train_mlp(X_train_fe, X_test_fe, y_train, y_test)

CV scores: 0.8335 +/- 0.0082
Balanced Accuracy: 0.8227


### `X_train_poly`

In [165]:
mlp_poly, mlp_preds_poly = train_mlp(X_train_poly, X_test_poly, y_train, y_test)

CV scores: 0.6207 +/- 0.0056
Balanced Accuracy: 0.6269


## XGBoost

In [120]:
def train_xgboost(X_tr, X_te, y_tr, y_te, params = {}):
    xgb = XGBClassifier(
    random_state=SEED,
    enable_categorical=True,
    device="cuda:0",
    tree_method="hist",
    **params
)

    xgb_cvs = cross_val_score(xgb, X_tr, y_tr, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
    print(f"cv scores: {xgb_cvs.mean():.4f} +/- {xgb_cvs.std():.4f}")

    xgb.fit(X_tr, y_tr)
    xgb_preds = xgb.predict(X_te)

    xgb_metric_score = balanced_accuracy_score(y_te, xgb_preds)
    print(f"Balanced Accuracy: {xgb_metric_score:.4f}")

    return xgb, xgb_preds

### `X_train`

In [121]:
xgb_base, xgb_preds_base = train_xgboost(X_train, X_test, y_train, y_test)

cv scores: 0.9615 +/- 0.0011
Balanced Accuracy: 0.9647


### `X_train_fe`

In [122]:
xgb_fe, xgb_preds_fe = train_xgboost(X_train_fe, X_test_fe, y_train, y_test)

cv scores: 0.8460 +/- 0.0027
Balanced Accuracy: 0.8471


### `X_train_poly`

In [123]:
xgb_poly, xgb_preds_poly = train_xgboost(X_train_poly, X_test_poly, y_train, y_test)

cv scores: 0.6550 +/- 0.0025
Balanced Accuracy: 0.6577


# Evaluate Feature Importances

- look at gain, permutation, and shap for the best performing model for each feature set

In [173]:
xgb_base_imp = get_importances(xgb_base, X_test, y_test, "xgb_base", has_builtin=True)
lgbm_base_imp  = get_importances(lgbm_base, X_test, y_test, "lgbm_base", has_builtin=False)
mlp_base_imp  = get_importances(mlp_base, X_test, y_test, "mlp_base", has_builtin=False)

combined = xgb_base_imp.merge(lgbm_base_imp, on="feature")
combined = combined.merge(mlp_base_imp, on="feature")
combined

,feature,xgb_base_perm,xgb_base_rank_perm,xgb_base_gain,xgb_base_rank_gain,lgbm_base_perm,lgbm_base_rank_perm,mlp_base_perm,mlp_base_rank_perm
0,Soil_Type,-0.000040,19.0,0.001291,17.0,0.000014,14.0,-0.000141,14.0
1,Soil_pH,0.000010,15.0,0.001394,15.0,0.000046,12.0,-0.000327,19.0
2,Soil_Moisture,0.218529,2.0,0.115981,3.0,0.218848,2.0,0.217930,2.0
3,Organic_Carbon,0.000090,9.0,0.001406,13.0,0.000028,13.0,-0.000050,9.0
4,Electrical_Conductivity,0.000033,12.0,0.001401,14.0,0.000083,10.0,-0.000185,18.0
5,Temperature_C,0.111810,5.0,0.072006,5.0,0.111974,5.0,0.108301,4.0
6,Humidity,0.000145,8.0,0.002015,8.0,0.000127,8.0,-0.000077,10.0
7,Rainfall_mm,0.021029,6.0,0.025715,6.0,0.020775,6.0,0.022313,6.0
8,Sunlight_Hours,-0.000039,18.0,0.001449,11.0,0.000085,9.0,0.000007,8.0
9,Wind_Speed_kmh,0.113864,4.0,0.086152,4.0,0.113855,4.0,0.106517,5.0


# Prune Features?

Given that each model liked the base set the best, which only has 19 features. I will keep all features.

# Tuning

- tune the best model for each architecture

In [181]:
def objective_xgb(trial):
    # Tree structure
    max_depth = trial.suggest_int("max_depth", 3, 12)
    min_child_weight = trial.suggest_float("min_child_weight", 1, 20, log=True)
    min_split_loss = trial.suggest_float("min_split_loss", 1e-8, 5.0, log=True)  # gamma

    # Learning rate & boosting rounds
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)

    # Sampling (helps a lot with redundant polynomial features)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0)
    colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.5, 1.0)

    # Regularization (important when polynomial features create multicollinearity)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True)   # L1
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True) # L2

    cls_optuna = XGBClassifier(
        random_state=SEED,
        enable_categorical=True,
        device="cuda:0",
        tree_method="hist",
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        min_split_loss=min_split_loss,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        colsample_bylevel=colsample_bylevel,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
    )

    cv_scores = cross_val_score(
        cls_optuna, X_train, y_train,
        cv=skf, scoring='balanced_accuracy', n_jobs=1
    )
    return cv_scores.mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=50)

[I 2026-04-30 00:58:02,487] A new study created in memory with name: no-name-898b08f3-0c49-4761-97ae-6ed81c9208c3
[I 2026-04-30 00:58:30,909] Trial 0 finished with value: 0.8691882166313581 and parameters: {'max_depth': 7, 'min_child_weight': 6.978685633786163, 'min_split_loss': 0.7539321735403094, 'learning_rate': 0.028969079717321256, 'n_estimators': 250, 'subsample': 0.6557888937953722, 'colsample_bytree': 0.35917671368872406, 'colsample_bylevel': 0.8422797843719144, 'reg_alpha': 3.8417035043968886, 'reg_lambda': 1.9503183058660278}. Best is trial 0 with value: 0.8691882166313581.
[I 2026-04-30 00:58:45,421] Trial 1 finished with value: 0.960981314772859 and parameters: {'max_depth': 5, 'min_child_weight': 14.241367818934041, 'min_split_loss': 4.677086126810762e-08, 'learning_rate': 0.12732147444397943, 'n_estimators': 150, 'subsample': 0.5546557801116492, 'colsample_bytree': 0.8714880619337733, 'colsample_bylevel': 0.9433579761558848, 'reg_alpha': 0.0059872339560452965, 'reg_lambda

In [190]:
def objective_mlp(trial):
    # Architecture
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layer_size = trial.suggest_int("layer_size", 32, 256, step=32)
    # Funnel architecture: each layer halves (common, effective default)
    hidden_layer_sizes = tuple(max(16, layer_size // (2**i)) for i in range(n_layers))

    # Activation
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])

    # Optimizer & learning rate
    learning_rate = trial.suggest_categorical("learning_rate", ["adaptive", "invscaling", "constant"])
    solver = trial.suggest_categorical("solver", ["adam"])  # adam is almost always best for this
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)

    # Regularization (critical for polynomial features)
    alpha = trial.suggest_float("alpha", 1e-6, 1e-1, log=True)  # L2 penalty

    # Batch size
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])

    # Training control
    max_iter = trial.suggest_int("max_iter", 100, 500, step=50)

    # Early stopping (helps prevent overfitting on noisy poly features)
    early_stopping = True
    validation_fraction = 0.1
    n_iter_no_change = 15

    mlp_optuna = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        learning_rate_init=learning_rate_init,
        alpha=alpha,
        learning_rate=learning_rate,
        batch_size=batch_size,
        max_iter=max_iter,
        early_stopping=early_stopping,
        validation_fraction=validation_fraction,
        n_iter_no_change=n_iter_no_change,
        random_state=SEED,
    )

    mlp_pipe = Pipeline(
        [
            ("scale", ColumnTransformer(transformers=[
               ("std", StandardScaler(), make_column_selector(dtype_include="number"))
            ])),
            ("model", mlp_optuna)
        ]
    )

    mlp_pipe.fit(X_train, y_train)
    y_preds = mlp_pipe.predict(X_test)

    return balanced_accuracy_score(y_test, y_preds)

study_mlp = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5) # early stop bad trials
)
study_mlp.optimize(objective_mlp, n_trials=5)

print("Best params:", study_mlp.best_params)
print("Best score:", study_mlp.best_value)

[I 2026-04-30 02:26:14,975] A new study created in memory with name: no-name-cd5ecd35-0bed-4911-b8df-25d77caf2b9d
[I 2026-04-30 02:27:10,669] Trial 0 finished with value: 0.6631428131344276 and parameters: {'n_layers': 1, 'layer_size': 128, 'activation': 'relu', 'learning_rate': 'adaptive', 'solver': 'adam', 'learning_rate_init': 0.004536773873650507, 'alpha': 0.0001536581739916497, 'batch_size': 256, 'max_iter': 350}. Best is trial 0 with value: 0.6631428131344276.
[I 2026-04-30 02:30:48,904] Trial 1 finished with value: 0.636332818280192 and parameters: {'n_layers': 3, 'layer_size': 224, 'activation': 'relu', 'learning_rate': 'adaptive', 'solver': 'adam', 'learning_rate_init': 0.003321291235699812, 'alpha': 9.998404561567452e-06, 'batch_size': 64, 'max_iter': 500}. Best is trial 0 with value: 0.6631428131344276.
[I 2026-04-30 02:33:47,095] Trial 2 finished with value: 0.6304835130479814 and parameters: {'n_layers': 3, 'layer_size': 64, 'activation': 'relu', 'learning_rate': 'adaptive

Best params: {'n_layers': 1, 'layer_size': 128, 'activation': 'relu', 'learning_rate': 'adaptive', 'solver': 'adam', 'learning_rate_init': 0.004536773873650507, 'alpha': 0.0001536581739916497, 'batch_size': 256, 'max_iter': 350}
Best score: 0.6631428131344276


In [187]:
def objective_lgbm(trial):
    # Tree structure
    num_leaves = trial.suggest_int("num_leaves", 15, 255)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    min_child_samples = trial.suggest_int("min_child_samples", 5, 100)

    # Learning
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)

    # Regularization
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True)   # L1
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True) # L2
    min_split_gain = trial.suggest_float("min_split_gain", 1e-8, 1.0, log=True)

    # Sampling (helps prevent overfitting)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    subsample_freq = trial.suggest_int("subsample_freq", 0, 7)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)

    # Class imbalance handling
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    lgbm_optuna = LGBMClassifier(
        num_leaves=num_leaves,
        max_depth=max_depth,
        min_child_samples=min_child_samples,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        min_split_gain=min_split_gain,
        subsample=subsample,
        subsample_freq=subsample_freq,
        colsample_bytree=colsample_bytree,
        class_weight=class_weight,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )

    cv_scores = cross_val_score(
        lgbm_optuna, X_train, y_train,
        cv=skf, scoring='balanced_accuracy', n_jobs=-1
    )
    return cv_scores.mean()


study_lgbm = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study_lgbm.optimize(objective_lgbm, n_trials=25)

print("Best params:", study_lgbm.best_params)
print("Best score:", study_lgbm.best_value)

[I 2026-04-30 02:04:02,700] A new study created in memory with name: no-name-7e1d9787-1859-45ac-92c5-81e8c77a590d
[I 2026-04-30 02:05:17,862] Trial 0 finished with value: 0.9604625460088638 and parameters: {'num_leaves': 162, 'max_depth': 11, 'min_child_samples': 60, 'learning_rate': 0.026616005911258348, 'n_estimators': 400, 'reg_alpha': 0.016634274222109485, 'reg_lambda': 0.0002573758245271361, 'min_split_gain': 0.0001623561499991566, 'subsample': 0.581567887567268, 'subsample_freq': 7, 'colsample_bytree': 0.8534145857091512, 'class_weight': None}. Best is trial 0 with value: 0.9604625460088638.
[I 2026-04-30 02:05:40,621] Trial 1 finished with value: 0.9594975968140323 and parameters: {'num_leaves': 144, 'max_depth': 12, 'min_child_samples': 34, 'learning_rate': 0.08767670397855803, 'n_estimators': 100, 'reg_alpha': 4.191444162777437e-05, 'reg_lambda': 1.1799352902477585, 'min_split_gain': 1.5019785193818356e-05, 'subsample': 0.5697355934236379, 'subsample_freq': 7, 'colsample_bytre

Best params: {'num_leaves': 209, 'max_depth': 3, 'min_child_samples': 95, 'learning_rate': 0.1266407030576051, 'n_estimators': 450, 'reg_alpha': 0.0004341634743646484, 'reg_lambda': 4.992980639893465e-05, 'min_split_gain': 0.0011667766404297238, 'subsample': 0.700149479371766, 'subsample_freq': 2, 'colsample_bytree': 0.6078309990280749, 'class_weight': 'balanced'}
Best score: 0.9713758052192372


# Stacking

In [196]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Rebuild each base model with its best params from Optuna
# (assumes you've run all three studies)

# --- LightGBM ---
best_lgbm = LGBMClassifier(
    **study_lgbm.best_params,
    random_state=SEED,
    n_jobs=1,          # set to 1 — StackingClassifier parallelizes across estimators
    verbose=-1,
)

# --- XGBoost ---
best_xgb = XGBClassifier(
    **study_xgb.best_params,
    random_state=SEED,
    enable_categorical=True,
    n_jobs=1,
    eval_metric="logloss",  # adjust if needed
)

# --- MLP (needs scaling, so wrap in its own pipeline) ---
# Reconstruct the architecture the same way the objective did
n_layers = study_mlp.best_params["n_layers"]
layer_size = study_mlp.best_params["layer_size"]
hidden_layer_sizes = tuple(max(16, layer_size // (2**i)) for i in range(n_layers))

mlp_params = {k: v for k, v in study_mlp.best_params.items() if k not in ("n_layers", "layer_size")}

best_mlp = MLPClassifier(
    hidden_layer_sizes=hidden_layer_sizes,
    **mlp_params,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=SEED,
)

mlp_pipe = Pipeline([
    ("scale", ColumnTransformer(transformers=[
        ("ohe", OneHotEncoder(sparse_output=False), make_column_selector(dtype_include="category")),
        ("std", StandardScaler(), make_column_selector(dtype_include="number"))
    ], remainder="passthrough")),
    ("model", best_mlp),
])

# --- Stacking ---
estimators = [
    ("lgbm", best_lgbm),
    ("xgb", best_xgb),
    ("mlp", mlp_pipe),
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    stack_method="predict_proba",  # use probabilities as meta-features
    cv=skf,                         # same CV used during tuning — prevents leakage
    n_jobs=-1,
    passthrough=False,              # set True to also feed raw features to meta-learner
)

# Cross-validated score on training set
stack_cv_scores = cross_val_score(
    stack, X_train, y_train,
    cv=skf, scoring="balanced_accuracy", n_jobs=-1
)
print(f"Stacking CV balanced accuracy: {stack_cv_scores.mean():.4f} ± {stack_cv_scores.std():.4f}")

Stacking CV balanced accuracy: 0.9655 ± 0.0014


# Results & Submission

In [197]:
# load training data
test_df = pd.read_csv(DATA_DIR / "test.csv", index_col="id")

cat_cols = test_df.select_dtypes(include=["str", "object"]).columns
test_df[cat_cols] = test_df[cat_cols].astype("category")

test_df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [198]:
# fit on full dataset
stack.fit(X, y_encoded)

# get and check preds
y_preds = pd.DataFrame(le.inverse_transform(stack.predict(test_df)), index=test_df.index)
y_preds.head()

,0
id,
630000,Low
630001,Low
630002,Low
630003,Low
630004,Low


In [201]:
y_preds.to_csv("../../data/kaggle-irrigation/stacking_preds.csv")

# Discussion

**What did you try? What worked well? What didn't work well?**

I used two boosted-tree models and a neural network. I originally tried using `TabNet` but it took ages and performance was not too good. Maybe I was doing something wrong. The trees will likely amplify each other but the neural network complements them by approximating a function rather than making splits. The trees, as always, worked great. The neural network performed unexpectedly well: I'm not sure what I did different from the last time I used the `MLP` but it had previously liked polynomial features. This time it preferred the base data. I was really surprised that in every case, feature engineering harmed performance.

As an additional note, I recall in class we decided that, under a time constraint tuning beats feature engineering. Well, since feature engineering didnt work out, this is kind of a test drive.

**What will you continue to use moving forward?**

Gradient-Boosted Trees!! Given time constraints, I once again was unable to take a stab at using clusters as a feature. I will do this in the future. I will try entirely different models for stacking as well. I also didn't get a chance to use `shap` because it didn't play nicely with scikit-learn's `Pipeline()` . I think developing a more robust technique for pruning features and feeding different models their preferred features would go a long way.

**Were model improvements meaningful or small? Make sure to use performance metrics in your discussion.**

- XGB and LightGBM improvements from tuning were the best I've been able to achieve yet! `balanced_accuracy` increased to over 0.97 for the LightGBM and for XGB we saw a score of 0.962
- MLP decreased significantly. This is likely because it didn't have enough training rounds due to time constraints.

**Also provide leaderboard scores for the approaches you tried. A table might be useful for organizing the metrics and leaderboard scores.**

| Model       | Leaderboard Place | Balanced Accuracy Score | HW Created |
|-------------|-------------------|-------------------------|------------|
| `best_rf`   | -                 | 0.94561                 | 2          |
| `best_cb`   | 1801              | 0.95586                 | 2          |
| `final_xgb` | 1766              | 0.96013                 | 3          |
| `stack`     | 2038              | 0.96369                 | 4          |

